In [5]:
import pandas as pd
import numpy as np
import pygeohash as pgh
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
TRAIN_PATH = ""
TEST_PATH = ""

In [6]:
def preprocess_data(df, is_train=True):
    df = df.copy()
    
    # 1. Temporal Features (Extracting Hour, Minute, and Cyclical Time)
    df[['hour', 'minute']] = df['timestamp'].str.split(':', expand=True).astype(int)
    df['total_minutes'] = df['hour'] * 60 + df['minute']
    
    # Cyclical encoding for time (so 23:59 and 00:01 are mathematically close)
    df['time_sin'] = np.sin(2 * np.pi * df['total_minutes'] / 1440)
    df['time_cos'] = np.cos(2 * np.pi * df['total_minutes'] / 1440)
    
    # 2. Impute Missing Values intelligently
    # Impute Temperature based on the median of that specific day and hour
    df['Temperature'] = df.groupby(['day', 'hour'])['Temperature'].transform(lambda x: x.fillna(x.median()))
    df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median()) # Fallback
    
    # Treat missing categorical values as a distinct "Unknown" category
    df['RoadType'] = df['RoadType'].fillna('Unknown')
    df['Weather'] = df['Weather'].fillna('Unknown')
    
    # 3. Spatial Features (Decoding Geohash)
    # This transforms the categorical geohash into continuous GPS coordinates
    df['lat'] = df['geohash'].apply(lambda x: pgh.decode(x)[0])
    df['lon'] = df['geohash'].apply(lambda x: pgh.decode(x)[1])
    
    # 4. Feature formatting
    # Binarize boolean-like columns
    df['LargeVehicles'] = df['LargeVehicles'].map({'Allowed': 1, 'Not Allowed': 0})
    df['Landmarks'] = df['Landmarks'].map({'Yes': 1, 'No': 0})
    
    # Convert remaining strings to Categorical for LightGBM
    cat_cols = ['RoadType', 'Weather', 'geohash']
    for c in cat_cols:
        df[c] = df[c].astype('category')
        
    # Drop unneeded columns
    cols_to_drop = ['Index', 'timestamp', 'total_minutes']
    df.drop(columns=cols_to_drop, inplace=True)
    
    return df

In [ ]:
print("Loading data...")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

# Note: test_demand.csv actually contains a 'demand' column which we need to predict/evaluate against.
print("Preprocessing data...")
train_clean = preprocess_data(train)
test_clean = preprocess_data(test)

# Separate features and target
X_train = train_clean.drop(columns=['demand'])
y_train = train_clean['demand']

X_test = test_clean.drop(columns=['demand'])
y_test = test_clean['demand']

# Setup LightGBM Parameters
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 127,
    'max_depth': 10,
    'feature_fraction': 0.8,
    'verbose': -1,
    'random_state': 42
}

Loading data...
Preprocessing data...


In [8]:
print("Training LightGBM model...")
# K-Fold Cross Validation for robust training
kf = KFold(n_splits=5, shuffle=True, random_state=42)
test_preds = np.zeros(len(X_test))
oof_preds = np.zeros(len(X_train))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**lgb_params, n_estimators=1500)
    
    # Train the model
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    
    # Predict Out-of-Fold and Test
    oof_preds[val_idx] = model.predict(X_va)
    test_preds += model.predict(X_test) / kf.n_splits
    print(f"Fold {fold+1} Finished.")

# Calculate Metrics
oof_r2 = r2_score(y_train, oof_preds)
test_r2 = r2_score(y_test, test_preds)

print("\n" + "="*40)
print(f"Validation R2 Score (OOF):  {oof_r2:.4f}")
print(f"Test Demand R2 Score:       {test_r2:.4f}")
print("="*40)


Training LightGBM model...
Fold 1 Finished.
Fold 2 Finished.
Fold 3 Finished.
Fold 4 Finished.
Fold 5 Finished.

Validation R2 Score (OOF):  0.9605
Test Demand R2 Score:       0.9127
